# DeBERTa Fine-Tuning — Team Morale Prediction from Headlines

Fine-tuning `microsoft/deberta-v3-base` (regression, `num_labels=1`) on real GDELT headlines labeled by Claude Haiku. The model predicts team morale on a 1–10 scale.

**Pipeline:** pre-training on synthetic data → fine-tuning with k-fold CV validation → final model on the full dataset. Input text: `f"{team}: " + " . ".join(headlines)` — the team name is part of the input (team perspective).

> ⚠️ **Environment:** training requires `transformers==4.44.0` (Python 3.12, venv `.venv-train`). Newer `transformers 5.x` causes NaN on small data. The cell below verifies the runtime version.

In [2]:
import sys, transformers
print(sys.executable)
print("transformers", transformers.__version__)

D:\PycharmProjects\Sports_prediction\.venv-train\Scripts\python.exe
transformers 4.44.0


## 1. Imports

In [3]:
import json
import pandas as pd
import numpy as np
from datasets import Dataset, Value
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

## 2. HuggingFace Authentication
`deberta-v3-base` is public — login is optional, skipped when `HF_TOKEN` is missing.

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN - deberta-v3-base is public, skipping login")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\jakub\.cache\huggingface\token
Login successful
Loged to Huggingface


## 3. Data Loading
We load `real_morale_dataset.json` and build the `text` column in the format `"{team}: headline . headline . ..."` and `label` = morale (1–10). **The text format must be identical at inference** (`llm_claude_morale.py`).

In [5]:
DATA_PATH = "../data/real_morale_dataset.json"
with open(DATA_PATH, encoding = "utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([
    {
        "text" : f"{ex['team']}: " + " . ".join(ex["headlines"]),
        "label" : float(ex["morale_score"])
    }
    for ex in raw
])

print(f"Loaded {len(df)} examples")
print(df[["text", "label"]].head(3))


Loaded 633 examples
                                                text  label
0  Arsenal: Micah Richards claims Arsenal would  ...    7.0
1  Arsenal: I will suffer a huge bout of man flu ...    3.0
2  Arsenal: Arsenal star hailed as  one of the fi...    6.0


### Data Quality Check
Check for missing `label` / `text` values before training.

In [6]:
print(f"NaN labels: {df['label'].isna().sum()} / {len(df)}")
print(f"NaN text: {df['text'].isna().sum()} / {len(df)}")

NaN labels: 0 / 633
NaN text: 0 / 633


### Cleaning
Drop rows with missing `text` or `label`.

In [7]:
df = df.dropna(subset=['label', 'text'])
print(f"After dropping {len(df)} examples")

After dropping 633 examples


### Label Normalization
Scale morale `[1,10] → [0.1,1.0]` (divide by `LABEL_MAX=10`) — stabilizes regression. In metrics and inference we multiply back ×10.

In [8]:
LABEL_MAX = 10.0
df['label'] = df['label'].astype(np.float32)/LABEL_MAX

### Label Distribution
Number of examples after cleaning and the morale distribution on the 1–10 scale.

In [9]:
print(f"Examples: {len(df)}")
print(f"Label range: {df['label'].min():.2f} – {df['label'].max():.2f}")
print(f"\nMorale distribution (1-10 scale):")
print((df['label'] * LABEL_MAX).astype(int).value_counts().sort_index())

Examples: 633
Label range: 0.10 – 0.90

Morale distribution (1-10 scale):
label
1      3
2     50
3     92
4    104
5     80
6    121
7    114
8     54
9     15
Name: count, dtype: int64


## 4. Tokenizer and Metrics
`deberta-v3-base` tokenizer (max_len 256). `compute_metrics` computes MAE/MSE plus **Pearson, Spearman and `pred_std`** — crucial, because MAE alone is misleading: the model can "spit out" the mean (~5) and reach MAE ≈ baseline (1.6) without learning anything. `spearman > 0` and `pred_std > 0` confirm real learning.

In [10]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
from scipy.stats import pearsonr, spearmanr

def tokenize(batch):
      return tokenizer(batch['text'], truncation=True, padding="max_length", max_length=256)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    pred_orig = predictions * LABEL_MAX
    labels_orig = labels * LABEL_MAX
    mse = np.mean((pred_orig - labels_orig) ** 2)
    mae = np.mean(np.abs(pred_orig - labels_orig))
    pred_std = float(np.std(pred_orig))
    if pred_std < 1e-6:
      pearson = spearman = 0.0
    else:
      pearson = float(pearsonr(pred_orig, labels_orig)[0])
      spearman = float(spearmanr(pred_orig, labels_orig)[0])
    return {"mse": float(mse), "mae": float(mae),
          "pearson": pearson, "spearman": spearman, "pred_std": pred_std}

D:\PycharmProjects\Sports_prediction\.venv-train\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
D:\PycharmProjects\Sports_prediction\.venv-train\Lib\site-packages\transformers\convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


## 5. Pre-training on Synthetic Data
Sequential transfer: first we train the model on ~11k synthetic examples (`morale_dataset.json`) to give the regression head a sensible initialization. Saved to `deberta-morale-pretrained` (the cell skips training if the model already exists). Real data fine-tunes it in the next step.

In [16]:
import os
from sklearn.model_selection import train_test_split
from transformers import EarlyStoppingCallback


PRETRAINED_DIR = "./deberta-morale-pretrained"

if os.path.exists(PRETRAINED_DIR):
  print(f"Found pre-trained model at {PRETRAINED_DIR}, skipping pre-training.")
else:
  print("Loading synthetic dataset...")
  with open("../data/morale_dataset.json", encoding="utf-8") as f:
      syn_raw = json.load(f)

  syn_df = pd.DataFrame([
      {
          "text": f"{ex['team']}: " + " . ".join(ex["headlines"]),
          "label": np.float32(ex["morale_score"]) / LABEL_MAX
      }
      for ex in syn_raw
  ])

  syn_train, syn_val = train_test_split(syn_df, test_size=0.1, random_state=42)
  syn_train = syn_train.reset_index(drop=True)
  syn_val   = syn_val.reset_index(drop=True)
  print(f"Synthetic: {len(syn_train)} train, {len(syn_val)} val")

  syn_train_ds = Dataset.from_pandas(syn_train).map(tokenize, batched=True)
  syn_val_ds   = Dataset.from_pandas(syn_val).map(tokenize, batched=True)

  syn_train_ds = syn_train_ds.rename_column("label", "labels").cast_column("labels", Value("float32"))
  syn_val_ds   = syn_val_ds.rename_column("label", "labels").cast_column("labels", Value("float32"))

  syn_train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
  syn_val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

  pretrain_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

  pretrain_args = TrainingArguments(
      output_dir="./deberta-morale-pretrain-checkpoints",
      eval_strategy="epoch",
      save_strategy="epoch",
      metric_for_best_model="mae",
      greater_is_better=False,
      logging_steps=50,
      per_device_train_batch_size=32,
      per_device_eval_batch_size=32,
      num_train_epochs=3,
      warmup_ratio=0.1,
      weight_decay=0.01,
      learning_rate=2e-5,
      max_grad_norm=1.0,
      load_best_model_at_end=True,
      bf16=False,
      fp16=False,
  )

  pretrain_trainer = Trainer(
      model=pretrain_model,
      args=pretrain_args,
      train_dataset=syn_train_ds,
      eval_dataset=syn_val_ds,
      compute_metrics=compute_metrics,
      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
  )

  pretrain_trainer.train()
  pretrain_trainer.save_model(PRETRAINED_DIR)
  tokenizer.save_pretrained(PRETRAINED_DIR)
  print(f"Pre-trained model saved to {PRETRAINED_DIR}")

Znaleziono pre-trained model w ./deberta-morale-pretrained, pomijam pre-training.


## 6. Cross-Validation (5-fold CV)
Starting from the pre-trained model, fine-tune on real data across 5 folds. CV gives an **honest estimate of generalization** (mean MAE ± std) on a small dataset. Folds are for evaluation only — the final model is trained separately.

In [12]:
from sklearn.model_selection import KFold
import shutil

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
indices = np.arange(len(df))

fold_maes = []
best_mae = float("inf")
best_fold = -1

for fold, (train_idx, val_idx) in enumerate(kf.split(indices)):
    print(f"\n{'=' * 50}")
    print(f"Fold {fold + 1} / {N_FOLDS}")
    print(f"\n{'=' * 50}")

    train_df_fold = df.iloc[train_idx].reset_index(drop=True)
    val_df_fold = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = Dataset.from_pandas(train_df_fold).map(tokenize, batched = True)
    val_dataset = Dataset.from_pandas(val_df_fold).map(tokenize, batched = True)

    train_dataset = train_dataset.rename_column("label", "labels")
    val_dataset = val_dataset.rename_column("label", "labels")

    train_dataset = train_dataset.cast_column("labels", Value("float32"))
    val_dataset = val_dataset.cast_column("labels", Value("float32"))

    train_dataset.set_format(type="torch", columns= ["input_ids", "attention_mask", "labels"])
    val_dataset.set_format(type="torch", columns= ["input_ids", "attention_mask", "labels"])

    model = AutoModelForSequenceClassification.from_pretrained(PRETRAINED_DIR, num_labels=1)

    args = TrainingArguments(
          output_dir=f"./deberta-morale-fold{fold+1}",
          eval_strategy="epoch",
          save_strategy="epoch",
          metric_for_best_model="mae",
          greater_is_better=False,
          logging_steps=10,
          per_device_train_batch_size=32,
          per_device_eval_batch_size=32,
          num_train_epochs=15,
          warmup_ratio=0.1,
          weight_decay=0.01,
          learning_rate=2e-5,
          max_grad_norm=1.0,
          load_best_model_at_end=True,
          bf16=False,
          fp16=False,
      )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()

    metrics = trainer.evaluate()
    fold_mae = metrics["eval_mae"]
    fold_maes.append(fold_mae)
    print(f"Fold {fold+1} MAE: {fold_mae:.4f}")

    if fold_mae < best_mae:
          best_mae = fold_mae
          best_fold = fold + 1
          trainer.save_model("./deberta-morale-best")
          tokenizer.save_pretrained("./deberta-morale-best")
          print(f"  -> New best model! (fold {best_fold})")

print(f"\n{'='*50}")
print(f"K-FOLD RESULTS:")
for i, mae in enumerate(fold_maes):
    print(f"  Fold {i+1}: MAE = {mae:.4f}")
print(f"  Mean MAE: {np.mean(fold_maes):.4f} +/- {np.std(fold_maes):.4f}")
print(f"  Best fold: {best_fold} (MAE = {best_mae:.4f})")


Fold 1 / 5



Casting the dataset: 100%|██████████| 127/127 [00:00<00:00, 50649.10 examples/s]


Epoch,Training Loss,Validation Loss,Mse,Mae,Pearson,Spearman,Pred Std
1,0.038500,0.033453,3.345252,1.508505,0.344092,0.396693,0.453534
2,0.032100,0.034547,3.454683,1.536825,0.435697,0.460454,0.434923
3,0.033900,0.033047,3.304728,1.418943,0.511620,0.527996,0.822992
4,0.021500,0.021797,2.179683,1.142330,0.690159,0.703396,1.372986
5,0.012700,0.027143,2.714251,1.288503,0.699144,0.701664,1.419511
6,0.010100,0.016971,1.697070,0.964826,0.738516,0.733040,1.631992
7,0.006300,0.020605,2.060514,1.052215,0.706012,0.703906,1.737954
8,0.006200,0.024561,2.456149,1.206240,0.729000,0.724700,1.607382
9,0.004700,0.019710,1.971044,1.015598,0.711901,0.709466,1.740701


Fold 1 MAE: 0.9648
  -> New best model! (fold 1)

Fold 2 / 5



Casting the dataset: 100%|██████████| 127/127 [00:00<00:00, 126827.76 examples/s]


...

Epoch,Training Loss,Validation Loss,Mse,Mae,Pearson,Spearman,Pred Std
1,0.035700,0.037046,3.704584,1.589791,0.211449,0.228212,0.524966
2,0.034200,0.032459,3.245884,1.511892,0.307888,0.296213,0.513165
3,0.032700,0.030181,3.018099,1.421651,0.408217,0.380558,0.768216
4,0.023700,0.033255,3.325475,1.490542,0.526098,0.499614,1.232407
5,0.014500,0.027739,2.773939,1.247772,0.610355,0.564232,1.588027
6,0.009700,0.022588,2.258843,1.168908,0.631060,0.601263,1.535787
7,0.007900,0.022332,2.233219,1.159895,0.642298,0.603684,1.580895
8,0.006600,0.023052,2.305199,1.147717,0.632447,0.594248,1.566564
9,0.005100,0.023276,2.327580,1.167355,0.635414,0.607556,1.669675
10,0.004100,0.024077,2.407740,1.169657,0.638485,0.608397,1.643597


Fold 4 MAE: 1.1477

Fold 5 / 5



Casting the dataset: 100%|██████████| 126/126 [00:00<00:00, 126039.19 examples/s]


Epoch,Training Loss,Validation Loss,Mse,Mae,Pearson,Spearman,Pred Std
1,0.036200,0.046398,4.639777,1.797670,0.162601,0.160102,0.440684
2,0.033400,0.034763,3.476330,1.596120,0.254823,0.220305,0.399110
3,0.031800,0.033539,3.353867,1.496563,0.331394,0.308306,0.896501
4,0.018700,0.028462,2.846241,1.351013,0.527028,0.526470,1.200063
5,0.011900,0.026241,2.624063,1.289460,0.591717,0.575978,1.444876
6,0.009500,0.025353,2.535340,1.312471,0.647613,0.636864,1.427482
7,0.006000,0.022593,2.259335,1.244323,0.630977,0.612388,1.490969
8,0.005900,0.023798,2.379765,1.299698,0.661941,0.645303,1.400096
9,0.004900,0.021243,2.124332,1.208217,0.652990,0.625021,1.422385
10,0.003900,0.021523,2.152345,1.219895,0.661068,0.641975,1.444996


Fold 5 MAE: 1.2033

K-FOLD RESULTS:
  Fold 1: MAE = 0.9648
  Fold 2: MAE = 1.0055
  Fold 3: MAE = 1.1389
  Fold 4: MAE = 1.1477
  Fold 5: MAE = 1.2033
  Mean MAE: 1.0920 +/- 0.0909
  Best fold: 1 (MAE = 0.9648)


## 7. Final Model
Trained on **90% of the data with a 10% holdout** for early stopping (folds showed overfitting past the peak — a stop signal is needed). Starts from the pre-trained model. Saved to `models/deberta-morale-final` — this is the model the app loads. Result: **MAE ≈ 0.97, Spearman ≈ 0.80**.

In [17]:
import os
from sklearn.model_selection import train_test_split
from transformers import EarlyStoppingCallback

# 90/10 split — 10% holdout only for early stopping / best checkpoint selection
final_train_df, final_val_df = train_test_split(
  df, test_size=0.1, random_state=42, shuffle=True
)
final_train_df = final_train_df.reset_index(drop=True)
final_val_df   = final_val_df.reset_index(drop=True)
print(f"Final training: {len(final_train_df)} train, {len(final_val_df)} val")

ft_train = Dataset.from_pandas(final_train_df).map(tokenize, batched=True)
ft_val   = Dataset.from_pandas(final_val_df).map(tokenize, batched=True)
ft_train = ft_train.rename_column("label", "labels").cast_column("labels", Value("float32"))
ft_val   = ft_val.rename_column("label", "labels").cast_column("labels", Value("float32"))
ft_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
ft_val.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# start from the model pre-trained on synthetic data (NOT from scratch)
final_model = AutoModelForSequenceClassification.from_pretrained(PRETRAINED_DIR, num_labels=1)

final_args = TrainingArguments(
  output_dir="./deberta-morale-final-checkpoints",
  eval_strategy="epoch",
  save_strategy="epoch",
  metric_for_best_model="mae",
  greater_is_better=False,
  logging_steps=10,
  per_device_train_batch_size=32,
  per_device_eval_batch_size=32,
  num_train_epochs=15,
  warmup_ratio=0.1,
  weight_decay=0.01,
  learning_rate=2e-5,
  max_grad_norm=1.0,
  load_best_model_at_end=True,
  bf16=False,
  fp16=False,
)

final_trainer = Trainer(
  model=final_model,
  args=final_args,
  train_dataset=ft_train,
  eval_dataset=ft_val,
  compute_metrics=compute_metrics,
  callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

final_trainer.train()
final_metrics = final_trainer.evaluate()
print(f"\nFINAL model — MAE: {final_metrics['eval_mae']:.4f}, "
    f"Spearman: {final_metrics['eval_spearman']:.4f}, "
    f"pred_std: {final_metrics['eval_pred_std']:.4f}")

MODEL_OUTPUT = "../models/deberta-morale-final"
final_trainer.save_model(MODEL_OUTPUT)
tokenizer.save_pretrained(MODEL_OUTPUT)
print(f"Model saved to: {MODEL_OUTPUT}")

Final training: 569 train, 64 val


Casting the dataset: 100%|██████████| 64/64 [00:00<00:00, 63974.13 examples/s]


Epoch,Training Loss,Validation Loss,Mse,Mae,Pearson,Spearman,Pred Std
1,0.040100,0.049731,4.973079,1.829634,0.424735,0.494556,0.412738
2,0.035800,0.037110,3.710979,1.669003,0.537982,0.586617,0.304884
3,0.028500,0.029392,2.939190,1.438231,0.566155,0.553940,0.979217
4,0.019600,0.020765,2.076454,1.111742,0.749762,0.765902,1.404779
5,0.010800,0.016095,1.609522,0.970790,0.790315,0.802190,1.640133
6,0.008600,0.016858,1.685833,1.022767,0.782640,0.788327,1.629395
7,0.006700,0.016785,1.678484,0.971900,0.784543,0.788073,1.824074
8,0.005000,0.016493,1.649331,0.995409,0.792830,0.805291,1.598746



FINAL model — MAE: 0.9708, Spearman: 0.8022, pred_std: 1.6401
Model zapisany do: ../models/deberta-morale-final
